In [ ]:
#!/usr/bin/env python3
"""
Diurnal Hovmöller (LON vs local time) of precipitation efficiency for
THREE metrics (CWP, CIW, CLW): MCS and Non-MCS, plus percent difference,
with a lat-mean topography line overlaid.

UPDATED (manual vmin/vmax):
- Manually sets vmin/vmax per metric (CWP/CIW/CLW) for ε panels
- Forces MCS + Non-MCS to share the SAME color scale (per metric-row)
- Keeps ONE shared epsilon colorbar per row spanning MCS & Non-MCS
"""

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ==========================
# USER SETTINGS
# ==========================
basedir = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/"

FILES = {
    "CWP": {
        "mcs":     basedir + "Obs_cwp_PRECEFF_mcs_20160809-20160909_Asia.nc",
        "nonmcs":  basedir + "Obs_cwp_PRECEFF_non_mcs_20160809-20160909_Asia.nc",
        "label":   r"$\epsilon_{\mathrm{cwp}}$"
    },
    "CIW": {
        "mcs":     basedir + "Obs_ciw_PRECEFF_mcs_20160809-20160909_Asia.nc",
        "nonmcs":  basedir + "Obs_ciw_PRECEFF_non_mcs_20160809-20160909_Asia.nc",
        "label":   r"$\epsilon_{i}$"
    },
    "CLW": {
        "mcs":     basedir + "Obs_clw_PRECEFF_mcs_20160809-20160909_Asia.nc",
        "nonmcs":  basedir + "Obs_clw_PRECEFF_non_mcs_20160809-20160909_Asia.nc",
        "label":   r"$\epsilon_{\ell}$"
    },
}

f_topo = basedir + "topo_r1440x720_Asia_regridded.nc"

# Domain used in your Asia subset
LAT_MIN, LAT_MAX = -5, 40
LON_MIN, LON_MAX = 55, 115

UTC_TO_LOCAL_HOURS = 6
START_HOUR = 12

CLIP_MAX = 20          # clip huge eps outliers (h^-1)
PCTL_ABS = 95          # for %diff symmetric range
TOPO_YLIM_KM = (0, 6)

CMAP_EPS = "gist_earth_r"
CMAP_PCT = "RdBu_r"

# ---- MANUAL epsilon color limits (h^-1) ----
# (Edit these numbers to whatever you want)
MANUAL_EPS_LIMS = {
    "CWP": (0.0, 3),
    "CIW": (0.0, 5.0),
    "CLW": (0.0, 4.0),
}

# ---- OPTIONAL: manual percent-difference limits (%). If None, uses PCTL_ABS ----
MANUAL_PCT_LIM = 50.0   # set to None to use percentile-based limits

# ==========================
# HELPERS
# ==========================
def ensure_datetime_time(da):
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = xr.decode_cf(da.to_dataset(name="tmp"))["tmp"]
    return da

def make_diurnal_hovmoller(da, utc_to_local_hours=0, start_hour=12):
    """
    Input: da(time, x) where x is lon here
    Output: hov(hour_wrapped, x)
    """
    da = ensure_datetime_time(da)

    shift_seconds = int(np.round(utc_to_local_hours * 3600.0))
    da_local = da.copy()
    da_local["time"] = da_local["time"] + np.timedelta64(shift_seconds, "s")

    hod = da_local["time"].dt.hour
    hov = da_local.groupby(hod).mean("time", skipna=True)

    if "hour" not in hov.dims:
        for d in hov.dims:
            if d not in ["lon", "lat"] and hov[d].size == 24:
                hov = hov.rename({d: "hour"})
                break

    order = list(range(start_hour, 24)) + list(range(0, start_hour))
    hov = hov.sel(hour=order)
    return hov

def hour_label(h):
    h = int(h)
    if h == 0:  return "12AM"
    if h < 12:  return f"{h}AM"
    if h == 12: return "12PM"
    return f"{h-12}PM"

def edges_from_centers(x):
    x = np.asarray(x)
    if x.size < 2:
        raise ValueError("Need at least 2 coordinate points for edges.")
    dx = np.diff(x)
    edges = np.r_[x[0] - dx[0]/2, (x[:-1] + x[1:]) / 2, x[-1] + dx[-1]/2]
    return edges

def plot_hov_lon(ax, xcoord, hour_wrapped, field2d, cmap="viridis",
                 vmin=None, vmax=None, cbar_label="", add_colorbar=True,
                 xlabel="Longitude"):
    """
    field2d: (hour, xcoord) where xcoord = longitude centers
    """
    xcoord = np.asarray(xcoord)
    hour_wrapped = np.asarray(hour_wrapped)

    x_edges = edges_from_centers(xcoord)
    y_edges = np.arange(len(hour_wrapped) + 1)

    im = ax.pcolormesh(x_edges, y_edges, field2d, shading="auto",
                       cmap=cmap, vmin=vmin, vmax=vmax)

    yticks = [0, 6, 12, 18, 24]
    yticks = [t for t in yticks if t <= len(hour_wrapped)]
    ax.set_yticks(yticks)

    ylabels = []
    for t in yticks:
        if t == len(hour_wrapped):
            ylabels.append(hour_label(hour_wrapped[0]))
        else:
            ylabels.append(hour_label(hour_wrapped[t]))
    ax.set_yticklabels(ylabels)

    ax.tick_params(axis='y', labelsize=14)
    ax.tick_params(axis='x', labelsize=14)

    # dashed line at local midnight
    if 0 in list(hour_wrapped):
        idx0 = list(hour_wrapped).index(0)
        ax.axhline(idx0, ls="--", lw=1, color="white")

    if add_colorbar:
        cbar = plt.colorbar(im, ax=ax, pad=0.02)
        cbar.set_label(cbar_label, size=20)
        cbar.ax.tick_params(labelsize=20)

    ax.set_xlabel(xlabel, size=20)
    return im

def add_topography_line_lon(ax, lon, topo_km, color="black",
                            ylim_km=(0, 8), show_ylabel=True):
    """Overlay topography line with a right y-axis (topo vs longitude)."""
    ax2 = ax.twinx()
    ax2.plot(lon, topo_km, color=color, lw=3)
    ax2.set_ylim(*ylim_km)
    if show_ylabel:
        ax2.set_ylabel("Terrain (km)", color=color, size=20)
    else:
        ax2.set_yticklabels([])
    ax2.tick_params(axis="y", colors=color)
    ax2.spines["right"].set_color(color)
    ax2.tick_params(axis='y', labelsize=15)
    return ax2

def load_eps_latmean(path, lat_min, lat_max, lon_min, lon_max, clip_max=None):
    """
    Load PRECEFF, convert s^-1 -> h^-1, optional clip,
    then lat-mean -> (time, lon) for the specified lat band.
    """
    ds = xr.open_dataset(path, decode_times=True)
    da = ds["PRECEFF"] * 3600.0  # h^-1
    if clip_max is not None:
        da = da.where(da < clip_max)

    da_lat = da.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    ).mean("lat", skipna=True)

    return ds, da_lat

# ==========================
# LOAD + PROCESS TOPOGRAPHY (lat-mean topo vs lon)
# ==========================
dstopo = xr.open_dataset(f_topo)
topo2d = dstopo["topo"].where(dstopo["topo"] > -1e20)

topo_lon_km = topo2d.sel(
    lat=slice(LAT_MIN, LAT_MAX),
    lon=slice(LON_MIN, LON_MAX)
).mean("lat", skipna=True) / 1000.0

# ==========================
# PROCESS + PLOT
# ==========================
nrows = len(FILES)
ncols = 3

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 12), constrained_layout=True)

for r, (metric, info) in enumerate(FILES.items()):
    ds_m, da_m_lon = load_eps_latmean(info["mcs"], LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, clip_max=CLIP_MAX)
    ds_n, da_n_lon = load_eps_latmean(info["nonmcs"], LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, clip_max=CLIP_MAX)

    lon = ds_m["lon"].sel(lon=slice(LON_MIN, LON_MAX)).values

    # align topo lon to dataset lon (interp just in case)
    topo_row = topo_lon_km.interp(lon=ds_m["lon"].sel(lon=slice(LON_MIN, LON_MAX)))

    hov_m = make_diurnal_hovmoller(da_m_lon, utc_to_local_hours=UTC_TO_LOCAL_HOURS, start_hour=START_HOUR)
    hov_n = make_diurnal_hovmoller(da_n_lon, utc_to_local_hours=UTC_TO_LOCAL_HOURS, start_hour=START_HOUR)
    hour_wrapped = hov_m["hour"].values

    # % difference (protect divide by ~0)
    eps = 1e-12
    hov_pct = 100.0 * (hov_m - hov_n) / xr.where(np.abs(hov_n) > eps, hov_n, np.nan)

    # ---- MANUAL SAME COLOR SCALE for MCS + Non-MCS (per metric-row) ----
    vmin_row, vmax_row = MANUAL_EPS_LIMS[metric]

    # ---- %diff limits ----
    if MANUAL_PCT_LIM is None:
        vlim = np.nanpercentile(np.abs(hov_pct.values), PCTL_ABS)
        if not np.isfinite(vlim) or vlim == 0:
            vlim = 1.0
    else:
        vlim = float(MANUAL_PCT_LIM)

    eps_cbar_label = info["label"] + r" [h$^{-1}$]"

    # --------------------------
    # Column 1: MCS
    ax = axes[r, 0]
    im_m = plot_hov_lon(ax, lon, hour_wrapped, hov_m.values,
                        cmap=CMAP_EPS, vmin=vmin_row, vmax=vmax_row,
                        cbar_label="", add_colorbar=False,
                        xlabel="Longitude")
    ax.set_title("MCS", size=20)
    ax.set_ylabel("Local time", size=20)
    add_topography_line_lon(ax, lon, topo_row.values, ylim_km=TOPO_YLIM_KM, show_ylabel=False)

    # --------------------------
    # Column 2: Non-MCS
    ax = axes[r, 1]
    im_n = plot_hov_lon(ax, lon, hour_wrapped, hov_n.values,
                        cmap=CMAP_EPS, vmin=vmin_row, vmax=vmax_row,
                        cbar_label="", add_colorbar=False,
                        xlabel="Longitude")
    ax.set_title("Non-MCS", size=20)
    ax.set_ylabel("")
    add_topography_line_lon(ax, lon, topo_row.values, ylim_km=TOPO_YLIM_KM, show_ylabel=False)

    # Shared epsilon colorbar for col 1 & 2 (per row)
    cbar = fig.colorbar(im_n, ax=[axes[r, 0], axes[r, 1]], pad=0.02)
    cbar.set_label(eps_cbar_label, size=20)
    cbar.ax.tick_params(labelsize=20)

    # --------------------------
    # Column 3: % difference
    ax = axes[r, 2]
    im_p = plot_hov_lon(ax, lon, hour_wrapped, hov_pct.values,
                        cmap=CMAP_PCT, vmin=-vlim, vmax=vlim,
                        cbar_label=r"Δ(%) $\epsilon$", add_colorbar=True,
                        xlabel="Longitude")
    ax.set_ylabel("")
    ax.set_title("Percent difference", size=20)
    add_topography_line_lon(ax, lon, topo_row.values, ylim_km=TOPO_YLIM_KM, show_ylabel=True)

    # X-LABEL CONTROL (ONLY LAST ROW)
    for c in range(ncols):
        if r < nrows - 1:
            axes[r, c].set_xlabel("")
            axes[r, c].tick_params(axis="x", labelbottom=False)
        else:
            axes[r, c].set_xlabel("Longitude", size=20)
            axes[r, c].tick_params(axis="x", labelbottom=True)

    # optional sanity print
    print(f"{metric}: using manual ε limits vmin={vmin_row}, vmax={vmax_row} h^-1")

    ds_m.close()
    ds_n.close()

plt.show()
dstopo.close()

# OPTIONAL SAVE
fig.savefig('/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS5_Diurnal_Hovmöller_longitude.png',
             dpi=200, bbox_inches='tight')
fig.savefig('/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS5_Diurnal_Hovmöller_longitude.pdf',
             dpi=200, bbox_inches='tight')
